In [ ]:
# Electrode comparison
#
# Every setting (animals, probes, insertion protocol) lives in config.py;
# this notebook only runs the pipeline. Outputs: results/electrodes/ (CSVs)
# and results/figures/ (3-D renders).
import os
import pandas as pd
from PIL import Image

from config import (ANIMALS, ELECTRODES, RESULTS_DIR, SEG_CACHE_DIR, tiff_path,
                    Y_CENTER, POS_NUM, DEPTH_LIMIT,
                    FIG_X_CENTER, FIG_CROP_X, FIG_CROP_Y, FIG_FORMAT)
from Volume_bleeding import load_volume, process_cone_positions, find_first_black_pixel_slice
from Number_bleeding import load_segmentation, process_cone_positions_num
from model_3D_visualization import visualize_cone_pyvista, cropping_img

csv_dir = os.path.join(RESULTS_DIR, "electrodes")
fig_dir = os.path.join(RESULTS_DIR, "figures")
os.makedirs(csv_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

In [ ]:
# For each animal: load it once, then test every probe at POS_NUM sites.
# Writes Volume_<key>.csv and Number_<key>.csv per probe (all animals), plus
# one illustrative 3-D figure per probe per animal.

frames = {key: {"Volume": [], "Number": []} for key in ELECTRODES}

for animal in ANIMALS:
    path = tiff_path(animal)
    name = os.path.splitext(os.path.basename(path))[0]
    print(f"=== {name} ===")
    img = load_volume(path)                              # binary, for the volume metric
    seg = load_segmentation(path, SEG_CACHE_DIR, img)    # labels, for the count metric

    # The figure is rendered at one site (FIG_X_CENTER) and does not feed the CSVs.
    fig_start = find_first_black_pixel_slice(img, FIG_X_CENTER, Y_CENTER)
    fig_slab, cx, cy = cropping_img(img, FIG_X_CENTER, Y_CENTER, FIG_CROP_X, FIG_CROP_Y, fig_start)

    for key, cfg in ELECTRODES.items():
        print(f"  {cfg['label']}")
        geom = dict(shank_length=cfg["shank_length"],
                    shank_base_diameter=cfg["shank_base_d"], shank_top_diameter=cfg["shank_top_d"],
                    tip_length=cfg["tip_length"],
                    tip_base_diameter=cfg["tip_base_d"], tip_top_diameter=cfg["tip_top_d"])

        volume = process_cone_positions(img, Y_CENTER, **geom, depth_limit=DEPTH_LIMIT, pos_num=POS_NUM)
        number = process_cone_positions_num(seg, Y_CENTER, **geom, depth_limit=DEPTH_LIMIT, pos_num=POS_NUM)
        frames[key]["Volume"].append(pd.DataFrame({"file": name, "position": range(POS_NUM), "overlap_area": volume}))
        frames[key]["Number"].append(pd.DataFrame({"file": name, "position": range(POS_NUM), "overlap_number": number}))

        plotter = visualize_cone_pyvista(fig_slab, cx, cy, **geom, start_slice=fig_start,
                                         depth_limit=DEPTH_LIMIT, ui=0)
        fig_path = os.path.join(fig_dir, f"{key}_{name}.{FIG_FORMAT}")
        plotter.screenshot(fig_path)
        Image.open(fig_path).save(fig_path, dpi=(300, 300))   # stamp 300 dpi metadata

for key in ELECTRODES:
    for metric in ("Volume", "Number"):
        pd.concat(frames[key][metric], ignore_index=True).to_csv(
            os.path.join(csv_dir, f"{metric}_{key}.csv"), index=False)
print(f"Done. CSVs in {csv_dir}")